# H4 · GEOBench-VLM's xBD questions: our counting and our 7B on an outside benchmark

Roadmap H4 (O2), built 21 Nov 2026 at Abrar's go ("T2b, then H4"; the public Kaggle copy of xBD, checked against
the PC's copy). Kaggle GPU T4 x2, internet on, about 1.5 h.

**Why:** every counting number so far is on DisasterM3's own benchmark. GEOBench-VLM (ICCV 2025) asks 1,050
damaged-building counting questions on 210 xBD before-and-after pairs, and publishes GPT-4o at 56.67% and Qwen2-VL at
50.00% on them (its Table 3). It is an outside check of the claim behind our design: counting from a damage map
beats asking a VLM.

**The data:**
- **Questions:** GEOBench-VLM's xBD questions at a pinned revision (`b3485fe4`), saved in the `h4-inputs` dataset: 1,050
  counting questions (five per pair: no damage, minor, major, destroyed, damaged of any level) and 2 × 210 disaster
  type questions (after-photo alone; both photos).
- **Photos:** GEOBench doesn't ship xBD's photos. They come from a public Kaggle copy of full xBD
  (`qianlanzz/xbd-dataset`). **Copy check first:** its 5,598 training photos must be the same photos as the PC's copy
  of xBD: the same bytes, or else the same pixels once decoded (a PNG saved again with other compression settings
  differs in bytes but not in the photo; the smoke test found 28 of 5,598 byte-identical in the copy's `tier1/`).
  Every folder of the copy holding them is checked, and if no folder has all 5,598 with the same pixels, the run stops.
- **Overlap check** (the rule from 26 Oct: check by content, not names): every GEOBench photo is fingerprinted as E5 did
  (pixel hash, 64-bit difference hash, 32 × 32 grey thumbnail) against DisasterM3 Instruct's photos of the same ten
  events, the photos our 7B and R11 trained on. A pair is left out if either photo has the same pixels or a near copy
  there (difference hash within 6 bits and thumbnails correlating at 0.95 or more), or the same xBD name. Only clean
  pairs are scored.

**The two methods:**
- **Our counting (C4 on xBD photos):** R11 (U-Net, ResNet-50, after- and before-photo as 6 channels) with 8-view TTA,
  as in E7. The count is the number of 8-connected blobs of the asked class of at least 20 px, and the answer is the
  nearest option, as in E7. Our classes are intact, damaged and destroyed, so: no damage → intact; destroyed →
  destroyed. **Minor and major** have no class of their own (our damaged is both): they get the damaged count, a known
  mismatch, reported apart. **"Damaged, any level"** is worded as damaged buildings, but GEOBench's answer counts every
  building: on all 210 pairs it equals no damage + minor + major + destroyed (it equals the damaged ones alone on only
  58). So it gets our count of all buildings (blobs of any class), and it is reported apart, outside the rule (found
  in the third smoke test, checked on the question file, fixed before the full run).
- **Our 7B answering directly** (`AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP`): the DisasterM3 authors' prompts and
  scorer, as in L0 step 2 and L2 step 4 (images first, before then after; capital letter only; greedy), with GEOBench's
  first wording of each question. Each counting question is also asked **without its options**, for a number.

**Rules that never see the photo** (computed from the options, shown beside every score): the middle option, the
smallest option, and a rule that knows only the question type (the middle option for no damage and for damaged of any
level, the smallest for minor, major and destroyed). On all 1,050 questions they get 37.4%, 49.2% and 68.9%, against
GPT-4o's published 56.67%. GEOBench builds the options around the true count (within 20% and 40% of it), and the true
count is 0 in 49% of the questions (about 70% of the minor, major and destroyed ones). So accuracy on these options
can't show counting well, and the rule also compares the counts themselves.

**The rule, fixed before the run:** on the clean pairs' questions our classes can answer and whose answer means
what the question says (no damage, destroyed), our counting holds on this outside benchmark only if both:
1. its accuracy beats our 7B's by more than chance (McNemar's test, p < 0.05);
2. its count is closer to the true count than the 7B's own number (asked without options) more often than not (a sign
   test on the questions where they differ, p < 0.05). A question the 7B answers with no number counts against it.

Beside it: all five types, the counting errors, the rules that never see the photo, the disaster-type questions (7B
only), and the published scores, which use all 1,050 questions and no overlap check.

**Expectation, fixed before the run:**
- the copy check passes: 5,598 of 5,598 with the same pixels in one folder of the copy;
- the overlap isn't predicted (DisasterM3 built its Instruct set from xBD; which xBD photos it kept is what the check
  measures);
- our counting 35–55% on the answerable types, our 7B 25–45%: both tests of the rule pass;
- both below the question-type rule that never sees the photo (65.7% on these two types), and our counting below
  GPT-4o's published 56.67% on all five types (minor and major get the damaged count);
- our 7B at 55% or more on disaster type (68.3% on DisasterM3's own disaster-type task).

In [1]:
import ast, base64, glob, hashlib, json, os, re, shutil, socket, subprocess, sys, threading, time, urllib.request
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

SMOKE_TEST = False              # True: 10 pairs and their questions (about 25 min); False: everything (about 1.5 h)
GEOBENCH_REVISION = "b3485fe4971213ee13f91960638ddc5c4a098e70"
MODEL_REPO, MODEL_COMMIT = "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP", "def8f6c81889ffb68904afdc90c3a1389697d73b"
SERVED, PORT, MAX_MODEL_LEN = "disasterm3-7b", 8000, 8192
MIN_AREA = 20                   # E7's blobs: 8-connected, at least 20 px
NUM_CLASSES = 4                 # 0 background, 1 intact, 2 damaged, 3 destroyed
NEAR_DHASH, NEAR_CORR, DARK, FLAT = 6, 0.95, 0.5, 4.0    # E5 step 1's near-copy rule
OUT_DIR = Path("/kaggle/working/H4")
OUT_DIR.mkdir(parents=True, exist_ok=True)
T0 = time.time()
socket.create_connection(("pypi.org", 443), timeout=5).close()    # internet: segmentation models, vLLM, the 7B

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.3.4", "timm==0.9.7"],
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]      # E7's versions: R11's checkpoint key layout
import cv2
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
from scipy.stats import binomtest

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMNET_MEAN, IMNET_STD = np.array([0.485, 0.456, 0.406], np.float32) * 255, np.array([0.229, 0.224, 0.225], np.float32) * 255


def minutes():
    return f"[{(time.time() - T0) / 60:5.1f} min]"


print(minutes(), "torch", torch.__version__, "| smp", smp.__version__, "| GPUs", torch.cuda.device_count())

[  0.5 min] torch 2.11.0+cu130 | smp 0.3.4 | GPUs 2


In [2]:
# ── Inputs ──
def find(pattern, what):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{what} is missing: add it with + Add Input")
    return Path(hits[0])


QFILE = find("geobench_xbd_questions.json", "The h4-inputs dataset")
HFILE = find("xbd_train_sha256_pc.json", "The h4-inputs dataset")
UNET_CKPT = find("R11/final_model.pth", "abrarmohammedtanzim/disasterm3-r11-r9-checkpoints")
MANIFEST = find("train_release.json", "abrarmohammedtanzim/disasterm3-mirror")
XBD = find("xbd/hold/images", "qianlanzz/xbd-dataset").parent.parent       # .../xbd
Q = json.load(open(QFILE, encoding="utf-8"))
assert Q["revision"] == GEOBENCH_REVISION, Q["revision"]
QUESTIONS = Q["questions"]
PC_HASHES = json.load(open(HFILE, encoding="utf-8"))["sha256"]
print(minutes(), f"{len(QUESTIONS)} questions", Counter((q["part"], q["task"]) for q in QUESTIONS))
print("xBD copy:", XBD, "| splits:", sorted(p.name for p in XBD.iterdir() if p.is_dir()))

[  0.8 min] 1470 questions Counter({('Temporal', 'Damaged Building Counting'): 1050, ('Single', 'Disaster Type Classification'): 210, ('Temporal', 'Disaster Type Classification'): 210})
xBD copy: /kaggle/input/datasets/qianlanzz/xbd-dataset/xbd | splits: ['hold', 'test', 'tier1', 'tier3', 'train']


In [3]:
# ── The copy check: the public xBD copy's training photos against the PC's copy, byte for byte ──
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def pixel_md5(path):
    """E5 step 1's pixel hash: md5 of 'HxW:' and the decoded photo, so a PNG saved again with other settings matches."""
    cv2.setNumThreads(1)
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        return ""
    h, w = img.shape[:2]
    return hashlib.md5(f"{h}x{w}:".encode() + img.tobytes()).hexdigest()


PC_PIXELS = json.load(open(HFILE, encoding="utf-8"))["pixel_md5"]
known = sorted(PC_HASHES)[0]                    # every folder of the copy holding the PC's training photos
folders = sorted({Path(h).parent for h in glob.glob(f"{XBD}/**/{known}", recursive=True)})
CHECKS = []
for folder in folders:
    mirror = {p.name: p for p in folder.iterdir()}
    names = sorted(set(PC_HASHES) & set(mirror))
    with ThreadPoolExecutor(8) as ex:
        same_bytes = [h == PC_HASHES[n] for n, h in zip(names, ex.map(lambda n: sha256(mirror[n]), names))]
        same_pixels = [h == PC_PIXELS[n] for n, h in zip(names, ex.map(lambda n: pixel_md5(mirror[n]), names))]
    CHECKS.append({"folder": str(folder), "pc_files": len(PC_HASHES), "in_copy": len(names),
                   "same_bytes": int(sum(same_bytes)), "same_pixels": int(sum(same_pixels)),
                   "different_pixels": [n for n, ok in zip(names, same_pixels) if not ok][:10]})
    print(minutes(), "copy check:", CHECKS[-1])
COPY_CHECK = max(CHECKS, key=lambda c: (c["same_pixels"], c["same_bytes"])) if CHECKS else {"folder": None}
assert COPY_CHECK.get("same_pixels") == len(PC_HASHES), \
    "No folder of the public xBD copy has every training photo the same as the PC's copy: the run stops (see above)"
print("Passed:", COPY_CHECK["folder"], "has all", len(PC_HASHES), "training photos with the same pixels "
      f"({COPY_CHECK['same_bytes']} also with the same bytes)")

[  3.0 min] copy check: {'folder': '/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/images', 'pc_files': 5598, 'in_copy': 5598, 'same_bytes': 28, 'same_pixels': 5598, 'different_pixels': []}
[  3.8 min] copy check: {'folder': '/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/masks', 'pc_files': 5598, 'in_copy': 5598, 'same_bytes': 0, 'same_pixels': 0, 'different_pixels': ['guatemala-volcano_00000000_post_disaster.png', 'guatemala-volcano_00000000_pre_disaster.png', 'guatemala-volcano_00000001_post_disaster.png', 'guatemala-volcano_00000001_pre_disaster.png', 'guatemala-volcano_00000002_post_disaster.png', 'guatemala-volcano_00000002_pre_disaster.png', 'guatemala-volcano_00000006_post_disaster.png', 'guatemala-volcano_00000006_pre_disaster.png', 'guatemala-volcano_00000007_post_disaster.png', 'guatemala-volcano_00000007_pre_disaster.png']}
Passed: /kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/images has all 5598 training photos with the same pixels (28 also with th

In [4]:
# ── The 210 pairs: GEOBench's names (xBD_<name>) found in the copy's test and hold sets ──
def xbd_name(n):
    return n[4:] if n.startswith("xBD_") else n


TILES = {}
for q in QUESTIONS:
    names = q["image_name"] if isinstance(q["image_name"], list) else [q["image_name"]]
    for n in names:
        tile = re.sub(r"_(pre|post)_disaster\.png$", "", xbd_name(n))
        TILES.setdefault(tile, {})
WHERE = {}
for split in ("test", "hold", "train", "tier1", "tier3"):
    d = XBD / split / "images"
    if d.is_dir():
        for p in d.iterdir():
            WHERE.setdefault(p.name, p)
for tile, paths in TILES.items():
    for role in ("pre", "post"):
        p = WHERE.get(f"{tile}_{role}_disaster.png")
        if p is None:
            raise FileNotFoundError(f"{tile}_{role}_disaster.png is not in the xBD copy")
        paths[role] = p
print(minutes(), f"{len(TILES)} pairs found:", Counter(p['post'].parent.parent.name for p in TILES.values()))
if SMOKE_TEST:
    keep = sorted(TILES)[::max(1, len(TILES) // 10)][:10]
    TILES = {t: TILES[t] for t in keep}
    QUESTIONS = [q for q in QUESTIONS if re.sub(r"_(pre|post)_disaster\.png$", "", xbd_name(
        q["image_name"][0] if isinstance(q["image_name"], list) else q["image_name"])) in TILES]
    print(f"Smoke test: {len(TILES)} pairs, {len(QUESTIONS)} questions")

[  3.8 min] 210 pairs found: Counter({'test': 105, 'hold': 105})


In [5]:
# ── The overlap check against DisasterM3 Instruct: E5 step 1's fingerprints and near-copy rule, verbatim ──
def fingerprint(path):
    """One image file -> its fingerprints. Never raises: an unreadable file gets an empty pixel hash."""
    cv2.setNumThreads(1)
    raw = Path(path).read_bytes()
    out = {"path": str(path), "file_md5": hashlib.md5(raw).hexdigest(), "pixel_md5": "", "dhash": 0,
           "dark": 1.0, "h": 0, "w": 0, "thumb": np.zeros((32, 32), np.uint8)}
    img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR) if raw else None
    if img is None:
        return out
    h, w = img.shape[:2]
    grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(grey, (9, 8), interpolation=cv2.INTER_AREA).astype(np.int16)
    bits = (small[:, 1:] > small[:, :-1]).ravel()
    out.update(pixel_md5=hashlib.md5(f"{h}x{w}:".encode() + img.tobytes()).hexdigest(),
               dhash=int.from_bytes(np.packbits(bits).tobytes(), "big"),
               dark=round(float((grey < 8).mean()), 4), h=h, w=w,
               thumb=cv2.resize(grey, (32, 32), interpolation=cv2.INTER_AREA))
    return out


def bit_distance(a, b):
    """Hamming distance between uint64 values or arrays (broadcast)."""
    return np.bitwise_count(np.bitwise_xor(np.asarray(a, np.uint64), np.asarray(b, np.uint64))).astype(np.int16)


def thumb_corr(a, b):
    """Pearson correlation of paired rows of a and b (flattened thumbnails)."""
    a = a - a.mean(1, keepdims=True)
    b = b - b.mean(1, keepdims=True)
    den = np.sqrt((a * a).sum(1) * (b * b).sum(1))
    return np.where(den > 0, (a * b).sum(1) / np.maximum(den, 1e-9), 0.0)


def matchable(f):
    """E5's rule: images over half black, nearly flat or unreadable are never near-matched (dHash collides on them)."""
    return bool(f["pixel_md5"]) and f["dark"] <= DARK and float(f["thumb"].std()) >= FLAT


INS_ROOT = MANIFEST.parent / "train_images"            # E5 step 1's DisasterM3 Instruct photos: all of them
instruct = sorted(p for p in INS_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in (".png", ".jpg", ".jpeg", ".tif", ".tiff"))
if SMOKE_TEST:                                   # the smoke test: only the photos of its pairs' events
    events = {re.match(r"(.+?)_\d{8}$", t).group(1).replace("-", "_") for t in TILES}
    instruct = [p for p in instruct if any(e in p.name for e in events)]
with ThreadPoolExecutor(8) as ex:
    INS = list(ex.map(fingerprint, instruct))
print(minutes(), f"fingerprinted {len(INS)} DisasterM3 Instruct photos")
ins_md5 = defaultdict(list)
for f in INS:
    if f["pixel_md5"]:
        ins_md5[f["pixel_md5"]].append(Path(f["path"]).name)
ins_names = {Path(f["path"]).name for f in INS}
DH = np.array([f["dhash"] for f in INS], np.uint64)
TH = np.stack([f["thumb"].reshape(-1) for f in INS]).astype(np.float32)
OK = np.array([matchable(f) for f in INS])
OVERLAP = []
for tile, paths_ in TILES.items():
    for role, p in paths_.items():
        f = fingerprint(p)
        same = ins_md5.get(f["pixel_md5"], [])
        near = []
        if matchable(f):
            cand = np.where((bit_distance(DH, f["dhash"]) <= NEAR_DHASH) & OK)[0]
            if len(cand):
                c = thumb_corr(TH[cand], np.repeat(f["thumb"].reshape(1, -1).astype(np.float32), len(cand), 0))
                near = [Path(INS[i]["path"]).name for i, v in zip(cand, c) if v >= NEAR_CORR]
        named = p.name.replace("-", "_") in ins_names or p.name in ins_names     # xBD names use hyphens, DisasterM3 underscores
        if same or near or named:
            OVERLAP.append({"tile": tile, "role": role, "same_pixels": same[:3], "near_copies": near[:3], "same_name": named})
LEAKED = sorted({o["tile"] for o in OVERLAP})
CLEAN = sorted(set(TILES) - set(LEAKED))
print(minutes(), f"overlap: {len(LEAKED)} of {len(TILES)} pairs have a photo in DisasterM3 Instruct; {len(CLEAN)} clean")
for o in OVERLAP[:10]:
    print("  ", o)

[ 12.0 min] fingerprinted 20988 DisasterM3 Instruct photos
[ 12.7 min] overlap: 119 of 210 pairs have a photo in DisasterM3 Instruct; 91 clean
   {'tile': 'socal-fire_00000823', 'role': 'pre', 'same_pixels': ['socal_fire_00000823_pre_disaster.png'], 'near_copies': ['socal_fire_00000823_pre_disaster.png'], 'same_name': True}
   {'tile': 'socal-fire_00000823', 'role': 'post', 'same_pixels': ['socal_fire_00000823_post_disaster.png'], 'near_copies': ['socal_fire_00000823_post_disaster.png'], 'same_name': True}
   {'tile': 'socal-fire_00000179', 'role': 'pre', 'same_pixels': ['socal_fire_00000179_pre_disaster.png', 'woolsey_fire_00000732_pre_disaster.png'], 'near_copies': ['socal_fire_00000179_pre_disaster.png', 'woolsey_fire_00000732_pre_disaster.png'], 'same_name': True}
   {'tile': 'socal-fire_00000179', 'role': 'post', 'same_pixels': ['socal_fire_00000179_post_disaster.png'], 'near_copies': ['socal_fire_00000179_post_disaster.png'], 'same_name': True}
   {'tile': 'socal-fire_00001037', 

In [6]:
# ── Our counting: R11 with 8-view TTA (E7 step 2's code), blobs of at least 20 px ──
def read_rgb(path):
    image = cv2.imread(str(path))
    return None if image is None else cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def load_unet(path):
    """D4 checkpoint: U-Net + ResNet-50, 6 input channels (after-image, then before-image), 4 classes."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=6, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()


def unet_input(post, pre):
    """As R11's val_transform: ImageNet normalisation, channels 0-2 after-image, 3-5 before-image."""
    x = np.concatenate([(post.astype(np.float32) - IMNET_MEAN) / IMNET_STD,
                        (pre.astype(np.float32) - IMNET_MEAN) / IMNET_STD], 2)
    return torch.from_numpy(x.transpose(2, 0, 1).copy()).unsqueeze(0).to(DEVICE)


def _forward_once(model, x):
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    return model(x)[..., :h, :w].float()


TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]


def _tta_fwd(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x


def _tta_inv(y, k, flip):
    if flip:
        y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))


@torch.no_grad()
def predict_tta(model, x):
    prob_sum = 0
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        for k, flip in TTA_VIEWS:
            prob_sum = prob_sum + torch.softmax(_tta_inv(_forward_once(model, _tta_fwd(x, k, flip)), k, flip), dim=1)
    return (prob_sum / len(TTA_VIEWS)).argmax(1)[0].cpu().numpy().astype(np.uint8)


def blobs(binary):
    n, _, stats, _ = cv2.connectedComponentsWithStats(binary.astype(np.uint8), connectivity=8)
    return int((stats[1:n, cv2.CC_STAT_AREA] >= MIN_AREA).sum())


unet = load_unet(UNET_CKPT)
COUNTS = {}
for tile, p in TILES.items():
    post, pre = read_rgb(p["post"]), read_rgb(p["pre"])
    if pre.shape != post.shape:
        pre = cv2.resize(pre, (post.shape[1], post.shape[0]), interpolation=cv2.INTER_LINEAR)
    pred = predict_tta(unet, unet_input(post, pre))
    COUNTS[tile] = {"intact": blobs(pred == 1), "damaged": blobs(pred == 2), "destroyed": blobs(pred == 3),
                    "damaged_any": blobs((pred == 2) | (pred == 3)), "all_buildings": blobs(pred >= 1),
                    "px": [int((pred == c).sum()) for c in range(NUM_CLASSES)]}
del unet
torch.cuda.empty_cache()
print(minutes(), f"R11 with TTA on {len(COUNTS)} pairs; first: {next(iter(COUNTS.items()))}")

[ 14.9 min] R11 with TTA on 210 pairs; first: ('socal-fire_00000823', {'intact': 0, 'damaged': 0, 'destroyed': 0, 'damaged_any': 0, 'all_buildings': 0, 'px': [1048576, 0, 0, 0]})


In [7]:
# ── Each question's type, our counting's answer, and the no-image baselines ──
TYPES = [("no visible damage", "no damage", "intact"), ("minor damage", "minor", "damaged"),
         ("major damage", "major", "damaged"), ("completely destroyed", "destroyed", "destroyed"),
         ("regardless", "damaged, any level", "all_buildings")]    # its answer counts every building (first cell)


def qtype(prompt):
    for key, name, count_key in TYPES:
        if key in prompt:
            return name, count_key
    raise ValueError(f"unknown question type: {prompt}")


def nearest_option(n, options):
    return "ABCDE"[min(range(len(options)), key=lambda i: abs(options[i] - n))]


def tile_of(q):
    n = q["image_name"][0] if isinstance(q["image_name"], list) else q["image_name"]
    return re.sub(r"_(pre|post)_disaster\.png$", "", xbd_name(n))


rows = []
for i, q in enumerate(QUESTIONS):
    tile = tile_of(q)
    row = {"i": i, "part": q["part"], "task": q["task"], "tile": tile, "event": re.match(r"(.+?)_\d{8}$", tile).group(1),
           "clean": tile in CLEAN, "truth": q["ground_truth_option"], "question": q["prompts"][0]}
    if q["task"] == "Damaged Building Counting":
        name, key = qtype(q["prompts"][0])
        options = [float(x) for x in q["options_list"]]
        order = sorted(range(5), key=lambda j: options[j])
        row.update(type=name, answerable=name in ("no damage", "destroyed"), truth_count=float(q["ground_truth"]), options=options,
                   our_count=COUNTS[tile][key], our_answer=nearest_option(COUNTS[tile][key], options),
                   middle_answer="ABCDE"[order[2]], smallest_answer="ABCDE"[order[0]])
        row["type_rule_answer"] = row["middle_answer"] if name in ("no damage", "damaged, any level") else row["smallest_answer"]
        row["our_correct"] = row["our_answer"] == row["truth"]
        row["our_abs_error"] = abs(row["our_count"] - row["truth_count"])
    rows.append(row)
QDF = pd.DataFrame(rows)
COUNT_TYPES = {"our_correct": bool, "answerable": bool, "clean": bool, "truth_count": float, "our_count": float,
               "our_abs_error": float}


def counting(df):
    """The counting questions, with their columns as numbers (in the whole table they share rows with the others)."""
    return df[df.task == "Damaged Building Counting"].astype(COUNT_TYPES)


cnt = counting(QDF)
print(minutes(), "our counting, clean pairs:", cnt[cnt.clean].groupby("type")["our_correct"].mean().round(3).to_dict())

[ 14.9 min] our counting, clean pairs: {'damaged, any level': 0.088, 'destroyed': 0.593, 'major': 0.253, 'minor': 0.308, 'no damage': 0.132}


In [8]:
# ── Our 7B answering directly: vLLM on the two T4s, the DisasterM3 authors' prompts and scorer, greedy ──
CONSTRAINTS = find("pip_constraints.txt", "The h4-inputs dataset")      # T2 step 5's 107 packages, same versions
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", str(CONSTRAINTS), "vllm==0.25.1"],
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
from huggingface_hub import snapshot_download
model_dir = snapshot_download(MODEL_REPO, revision=MODEL_COMMIT, local_dir="/tmp/h4_7b")
cmd = [shutil.which("vllm") or "vllm", "serve", model_dir, "--served-model-name", SERVED, "--host", "127.0.0.1",
       "--port", str(PORT), "--tensor-parallel-size", "2", "--dtype", "half", "--enforce-eager",
       "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.92", "--max-num-seqs", "8",
       "--limit-mm-per-prompt", json.dumps({"image": 2}), "--mm-processor-kwargs", json.dumps({"max_pixels": 1344 * 1344})]
server = subprocess.Popen(cmd, stdout=open(OUT_DIR / "vllm_server.log", "w"), stderr=subprocess.STDOUT,
                          env=dict(os.environ, VLLM_NO_USAGE_STATS="1", DO_NOT_TRACK="1"), start_new_session=True)
for _ in range(180):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5)
        break
    except OSError:
        assert server.poll() is None, open(OUT_DIR / "vllm_server.log").read()[-4000:]
        time.sleep(10)
print(minutes(), "vLLM is ready")

TWO = ("Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best "
       "option from the candidate options provided.\nQuestion: {prompt}\nOptions: {options_str}\nYour task is to respond "
       "with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.")
ONE = ("Analyze the image to answer the following question. Choose the best option from the candidate options "
       "provided.\nQuestion: {prompt}\nOptions: {options_str}\nYour task is to respond with ONLY the capital letter of "
       "the correct option (e.g., C). Do not include any explanation or other text.")
OPEN = ("Analyze both the pre-disaster and post-disaster images to answer the following question.\nQuestion: {prompt}\n"
        "Your task is to respond with ONLY a whole number (e.g., 12). Do not include any explanation or other text.")


def letters(text):
    """The 499A scorer: the set of capital letters left after removing 'THE ANSWER IS' and 'OPTION'."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))


def image_url(path):
    return {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(open(path, "rb").read()).decode()}}


def ask(q, open_count=False):
    tile = tile_of(q)
    two = isinstance(q["image_name"], list)
    images = [TILES[tile]["pre"], TILES[tile]["post"]] if two else [TILES[tile]["post"]]
    text = (OPEN if open_count else TWO if two else ONE).format(prompt=q["prompts"][0], options_str=q["options"])
    body = {"model": SERVED, "temperature": 0, "max_tokens": 64 if open_count else 8, "seed": 0,   # a number may come in a sentence
            "messages": [{"role": "user", "content": [image_url(p) for p in images] + [{"type": "text", "text": text}]}]}
    for attempt in range(3):
        try:
            req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", json.dumps(body).encode(),
                                         {"Content-Type": "application/json"})
            return json.load(urllib.request.urlopen(req, timeout=300))["choices"][0]["message"]["content"]
        except OSError as e:
            last = e
            time.sleep(5)
    return f"ERROR {last}"


COUNT_Q = [q for q in QUESTIONS if q["task"] == "Damaged Building Counting"]
with ThreadPoolExecutor(8) as ex:
    answers = list(ex.map(ask, QUESTIONS))
    open_answers = list(ex.map(lambda q: ask(q, open_count=True), COUNT_Q))
QDF["vlm_answer"] = answers
QDF["vlm_correct"] = [letters(a) == {t} for a, t in zip(answers, QDF["truth"])]
first_int = lambda s: float(re.search(r"\d+", s).group(0)) if re.search(r"\d+", s or "") else float("nan")
is_count = QDF["task"] == "Damaged Building Counting"
QDF.loc[is_count, "vlm_open_answer"] = open_answers
QDF.loc[is_count, "vlm_count"] = [first_int(a) for a in open_answers]
QDF["vlm_abs_error"] = (QDF["vlm_count"] - QDF["truth_count"]).abs()
os.killpg(os.getpgid(server.pid), 15)
print(minutes(), f"7B: {len(answers)} answers and {len(open_answers)} numbers, "
      f"{sum(a.startswith('ERROR') for a in answers + open_answers)} errors, "
      f"{int(QDF.loc[is_count, 'vlm_count'].isna().sum())} counting questions with no number; accuracy by task:",
      QDF.groupby("task")["vlm_correct"].mean().round(3).to_dict())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

[ 22.2 min] vLLM is ready
[ 73.3 min] 7B: 1470 answers and 1050 numbers, 0 errors, 0 counting questions with no number; accuracy by task: {'Damaged Building Counting': 0.52, 'Disaster Type Classification': 0.552}


In [9]:
# ── Scores, the rule, and the files ──
PUBLISHED = {"Damaged Building Counting": {"GPT-4o": 0.5667, "Qwen2-VL": 0.5000, "LLaVA-OneVision": 0.4810, "EarthDial": 0.4362},
             "Disaster Type Classification (Temporal)": {"GPT-4o": 0.6300, "Qwen2-VL": 0.5991, "LLaVA-OneVision": 0.4537,
                                                         "EarthDial": 0.5727}}


def mcnemar(x, y):
    b, c = int((x & ~y).sum()), int((~x & y).sum())
    return b, c, (binomtest(b, b + c, 0.5).pvalue if b + c else 1.0)


cnt = counting(QDF).astype({"vlm_correct": bool, "vlm_count": float, "vlm_abs_error": float})
for rule in ("middle", "smallest", "type_rule"):
    cnt[f"{rule}_correct"] = cnt[f"{rule}_answer"] == cnt["truth"]
ans = cnt[cnt.clean & cnt.answerable]
b, c, p = mcnemar(ans["our_correct"].values, ans["vlm_correct"].values)
TEST1 = bool(ans["our_correct"].mean() > ans["vlm_correct"].mean() and p < 0.05)
# Test 2: whose count is closer to the true count (a missing 7B number counts against the 7B); ties left out
vlm_err = ans["vlm_abs_error"].fillna(float("inf"))
closer, farther = int((ans["our_abs_error"] < vlm_err).sum()), int((ans["our_abs_error"] > vlm_err).sum())
p2 = binomtest(closer, closer + farther, 0.5).pvalue if closer + farther else 1.0
TEST2 = bool(closer > farther and p2 < 0.05)
HOLDS = TEST1 and TEST2
cols = ["our_correct", "vlm_correct", "middle_correct", "smallest_correct", "type_rule_correct"]
by_type = cnt[cnt.clean].groupby("type")[cols].mean().round(4)
by_event = cnt[cnt.clean & cnt.answerable].groupby("event")[cols[:2]].mean().round(4)
errors = cnt[cnt.clean].groupby("type").agg(our_mae=("our_abs_error", "mean"), vlm_mae=("vlm_abs_error", "mean"),
                                            our_exact=("our_abs_error", lambda s: float((s == 0).mean())),
                                            vlm_exact=("vlm_abs_error", lambda s: float((s == 0).mean())),
                                            vlm_no_number=("vlm_count", lambda s: int(s.isna().sum()))).round(3)
dis = QDF[QDF.task == "Disaster Type Classification"].groupby(["part", "clean"])["vlm_correct"].agg(["mean", "size"]).round(4)
SUMMARY = {
    "smoke_test": SMOKE_TEST, "copy_check": COPY_CHECK, "pairs": len(TILES), "clean_pairs": len(CLEAN), "leaked_pairs": LEAKED,
    "answerable_clean_questions": int(len(ans)), "ours_answerable": round(float(ans["our_correct"].mean()), 4),
    "7b_answerable": round(float(ans["vlm_correct"].mean()), 4),
    "test1_accuracy_mcnemar": {"ours_only": b, "7b_only": c, "p": p, "passes": TEST1},
    "test2_closer_count_sign_test": {"ours_closer": closer, "7b_closer": farther, "p": p2, "passes": TEST2},
    "rule": "our counting holds" if HOLDS else "our counting does not hold",
    "type_rule_answerable": round(float(ans["type_rule_correct"].mean()), 4),
    "ours_all_five_clean": round(float(cnt[cnt.clean]["our_correct"].mean()), 4),
    "7b_all_five_clean": round(float(cnt[cnt.clean]["vlm_correct"].mean()), 4),
    "ours_all_five_all_pairs": round(float(cnt["our_correct"].mean()), 4),
    "7b_all_five_all_pairs": round(float(cnt["vlm_correct"].mean()), 4),
    "no_photo_rules_all_five_all_pairs": {r: round(float(cnt[f"{r}_correct"].mean()), 4) for r in ("middle", "smallest", "type_rule")},
    "published": PUBLISHED, "minutes": round((time.time() - T0) / 60, 1)}
print(json.dumps({k: v for k, v in SUMMARY.items() if k != "published"}, indent=1))
print(by_type.to_string(), "\n\n", errors.to_string(), "\n\n", by_event.to_string(), "\n\n", dis.to_string())
errors.to_csv(OUT_DIR / "count_errors.csv")
by_type.to_csv(OUT_DIR / "accuracy_by_type.csv")
QDF.to_csv(OUT_DIR / "per_question.csv", index=False)
pd.DataFrame(OVERLAP).to_csv(OUT_DIR / "overlap.csv", index=False)
pd.DataFrame([{"tile": t, **{k: v for k, v in c.items() if k != "px"}} for t, c in COUNTS.items()]).to_csv(OUT_DIR / "counts.csv", index=False)
(OUT_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=1, default=str), encoding="utf-8")
print(minutes(), "Wrote", sorted(p.name for p in OUT_DIR.iterdir()))

{
 "smoke_test": false,
 "copy_check": {
  "folder": "/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/images",
  "pc_files": 5598,
  "in_copy": 5598,
  "same_bytes": 28,
  "same_pixels": 5598,
  "different_pixels": []
 },
 "pairs": 210,
 "clean_pairs": 91,
 "leaked_pairs": [
  "hurricane-florence_00000033",
  "hurricane-florence_00000113",
  "hurricane-florence_00000205",
  "hurricane-florence_00000254",
  "hurricane-florence_00000305",
  "hurricane-florence_00000322",
  "hurricane-florence_00000511",
  "hurricane-florence_00000539",
  "hurricane-harvey_00000004",
  "hurricane-harvey_00000013",
  "hurricane-harvey_00000050",
  "hurricane-harvey_00000063",
  "hurricane-harvey_00000121",
  "hurricane-harvey_00000122",
  "hurricane-harvey_00000131",
  "hurricane-harvey_00000164",
  "hurricane-harvey_00000181",
  "hurricane-harvey_00000287",
  "hurricane-harvey_00000339",
  "hurricane-harvey_00000353",
  "hurricane-harvey_00000409",
  "hurricane-harvey_00000416",
  "hurricane-harvey

**After the run:** the outputs go into `Champion_Task/H4-results/H4/` (the smoke test into `H4-results/smoke/`), with a
README written from them, and the executed notebook over this one.